# Process a large file lazily

The pattern: read one record at a time, transform it, summarise the result. The whole point is that file size doesn't fit in memory but the *summary* does. The same approach handles 10 MB and 10 GB without changing — and it works on streams (stdin, network sockets) where there isn't a file size at all.

## The recipe in one line

```python
with open(path) as f:
    total = sum(parse(line) for line in f if relevant(line))
```

A file object is already an iterator — it yields one line at a time as you ask. A generator expression keeps the filter/transform lazy. A reducer (`sum`, `max`, `Counter`, writing to another file) consumes the stream end-to-end, holding only one line at a time in memory.

That's the whole pattern. The rest of this notebook unpacks the variations.

## Setup — make a sample file

We'll write a small CSV so we can demo with real I/O. (In production this would be the multi-GB file you can't load.)

In [ ]:
from pathlib import Path

sample = Path('/tmp/transactions.csv')
sample.write_text('''timestamp,user_id,category,amount
2024-01-15T09:23:11,42,food,12.50
2024-01-15T11:08:00,17,transport,3.20
2024-01-15T13:42:30,42,food,8.75
2024-01-16T07:15:00,99,utilities,45.00
2024-01-16T19:30:55,17,food,22.40
2024-01-17T08:01:00,42,transport,2.80
2024-01-17T20:14:20,17,utilities,30.00
2024-01-18T12:00:00,99,food,15.60
'''.strip() + '\n')

print(sample.read_text())

## Pattern 1 — line by line, with the file as iterator

`for line in f` and `(line for line in f)` both iterate the file lazily. The file's iterator yields one line per `next()` call, including the trailing newline.

In [ ]:
with open(sample) as f:
    next(f)              # skip the header
    total = sum(float(line.split(',')[3]) for line in f)

print(f'total: £{total:.2f}')

Memory used: a single line at a time. This works just as well on a 10-GB file.

## Pattern 2 — wrap in a generator function

Once the parsing logic is more than "split and cast one column", lift it into a generator function. You get a named, testable thing that takes a file handle and yields parsed records.

In [ ]:
import csv
from collections import namedtuple

Tx = namedtuple('Tx', 'timestamp user_id category amount')

def read_transactions(file):
    '''Yield Tx records lazily from an open CSV file.'''
    reader = csv.reader(file)
    next(reader)             # header
    for row in reader:
        yield Tx(
            timestamp=row[0],
            user_id=int(row[1]),
            category=row[2],
            amount=float(row[3]),
        )


with open(sample) as f:
    for tx in read_transactions(f):
        print(tx)

The generator does no work until you iterate. You can pass it through filters, splitters, aggregators — none of them block.

## Pattern 3 — chain multiple lazy stages

Filter, transform, summarise, all without materialising. This is the core "pipeline" idea.

In [ ]:
from collections import Counter

with open(sample) as f:
    transactions = read_transactions(f)
    food_only = (tx for tx in transactions if tx.category == 'food')
    by_user = Counter()
    for tx in food_only:
        by_user[tx.user_id] += tx.amount

for user, total in by_user.most_common():
    print(f'user {user}: £{total:.2f}')

Note: everything happens *inside* the `with` block. The generators only produce values while the file is open. If you returned them and the file got closed, subsequent `next()` would raise an error.

## Pattern 4 — chunked / record-spanning input

Sometimes a "record" is more than one line — a JSON object split across lines, a multi-line log entry, a fixed-size binary block. Wrap the file in a generator that yields whole records.

In [ ]:
log_text = '''2024-01-15 ERROR: connection refused
    at module foo.bar
    at module foo.baz
2024-01-15 INFO: retry succeeded
2024-01-16 ERROR: out of memory
    at module qux.quux
'''
log_path = Path('/tmp/multiline.log')
log_path.write_text(log_text)


def read_log_entries(file):
    '''Each entry starts with a date; lines beginning with whitespace are
    continuations.'''
    entry = []
    for line in file:
        if line and not line[0].isspace() and entry:
            yield ''.join(entry)
            entry = []
        entry.append(line)
    if entry:
        yield ''.join(entry)


with open(log_path) as f:
    for entry in read_log_entries(f):
        print('---')
        print(entry, end='')

The shape of `read_log_entries` is a useful template: accumulate lines into a buffer, yield at boundaries, flush at the end. It's still O(longest record) in memory, not O(file).

## Pattern 5 — fixed-size binary chunks with `iter(callable, sentinel)`

Two-argument `iter(callable, sentinel)` calls `callable()` repeatedly until it returns `sentinel`. It's the cleanest way to read a binary file in fixed-size blocks — no boilerplate `while True` loop.

In [ ]:
# Make a sample binary file
bin_path = Path('/tmp/data.bin')
bin_path.write_bytes(b'A' * 10 + b'B' * 10 + b'C' * 5)


with open(bin_path, 'rb') as f:
    for chunk in iter(lambda: f.read(8), b''):
        print(len(chunk), chunk)

When `f.read(8)` returns `b''` (end of file), `iter` stops. No conditional inside the loop body.

## Pattern 6 — write the output lazily too

If your output is also large — a transformed version of the input file, say — write it as you go rather than building a list and dumping at the end.

In [ ]:
out_path = Path('/tmp/transactions_uppercase.csv')

with open(sample) as fin, open(out_path, 'w', newline='') as fout:
    writer = csv.writer(fout)
    reader = csv.reader(fin)
    writer.writerow(next(reader))    # header passes through
    for row in reader:
        row[2] = row[2].upper()      # uppercase the category
        writer.writerow(row)


print(out_path.read_text())

Read one row, write one row. The whole transformation runs in constant memory.

## Anti-pattern: `f.readlines()` or `f.read()` on a large file

Both materialise the entire file in memory. Fine for a 10 KB config; catastrophic for a 10 GB log. If you ever find yourself writing them, ask whether a `for line in f` loop would do.

```python
# Bad — loads everything into memory
lines = open(path).readlines()
for line in lines:
    ...

# Good — one line at a time
with open(path) as f:
    for line in f:
        ...
```

## Why this matters beyond memory

Lazy reading also gives you:

- **Faster start-up** — your first result appears as soon as the first line is read, not after the whole file is loaded.
- **Streaming friendliness** — the same code works on stdin, a network socket, or a file. Anything iterable that yields records.
- **Composability** — each stage is independent and can be unit-tested with a small in-memory iterator.

The takeaway: file processing in Python is just iterator processing. The recipes you learnt for generators apply unchanged to "the data is on disk" or "the data is coming over the wire".